##Transform Constructors Data

1. Read bronze constructors table
2. Keep only the columns required for analytics (Drop url column)
3. Standardise column names using snake_case (constructorsId->constructors_id)
4. Rename columns to make them more meaningful (name->constructors_name)
5. Remove duplicate records
6. Transform values of columns nationality to Title Case
7. Write the transformed data to silver constructors table

In [0]:
dbutils.widgets.text("p_batch_id", "")
v_batch_id = dbutils.widgets.get("p_batch_id")

In [0]:
%run ../00-common/01.environment-config

In [0]:
%run ../00-common/03.silver-helpers

In [0]:
bronze_table = f"{catalog_name}.{bronze_schema}.constructors"
silver_table = f"{catalog_name}.{silver_schema}.constructors"

####Step 1: Read the bronze constructors table

In [0]:
constructors_df = (
    spark.table(bronze_table)
        .filter((F.col("batch_id") == v_batch_id))
)

In [0]:
display(constructors_df)

####Step 2: Keep only the coluumns required for analysis (Drop url column)

There are 2 ways to do this... one by selecting the required columns and the other by dropping the unwanted columns

In [0]:
##This method uses dropping of the non-required columns
from pyspark.sql import functions as F

constructors_dropped_df = constructors_df.drop("url")

display(constructors_dropped_df)

constructorId,name,nationality,ingestion_timestamp,source_file,batch_id
adams,Adams,american,2026-08-02T16:46:32.036Z,dbfs:/Volumes/formula1_incr/landing/files/2025-01/constructors.json,2025-01
afm,AFM,german,2026-08-02T16:46:32.036Z,dbfs:/Volumes/formula1_incr/landing/files/2025-01/constructors.json,2025-01
ags,AGS,french,2026-08-02T16:46:32.036Z,dbfs:/Volumes/formula1_incr/landing/files/2025-01/constructors.json,2025-01
alfa,Alfa Romeo,swiss,2026-08-02T16:46:32.036Z,dbfs:/Volumes/formula1_incr/landing/files/2025-01/constructors.json,2025-01
alphatauri,AlphaTauri,italian,2026-08-02T16:46:32.036Z,dbfs:/Volumes/formula1_incr/landing/files/2025-01/constructors.json,2025-01
alpine,Alpine F1 Team,french,2026-08-02T16:46:32.036Z,dbfs:/Volumes/formula1_incr/landing/files/2025-01/constructors.json,2025-01
alta,Alta,british,2026-08-02T16:46:32.036Z,dbfs:/Volumes/formula1_incr/landing/files/2025-01/constructors.json,2025-01
amon,Amon,new zealander,2026-08-02T16:46:32.036Z,dbfs:/Volumes/formula1_incr/landing/files/2025-01/constructors.json,2025-01
apollon,Apollon,swiss,2026-08-02T16:46:32.036Z,dbfs:/Volumes/formula1_incr/landing/files/2025-01/constructors.json,2025-01
arrows,Arrows,british,2026-08-02T16:46:32.036Z,dbfs:/Volumes/formula1_incr/landing/files/2025-01/constructors.json,2025-01


####Step 3,4: Standardizing Column names

 - Standardize column names using snake_case
 - Rename column names to make them meaningful

In [0]:
constructors_renamed_df = (
    constructors_dropped_df
        .withColumnsRenamed({
            "constructorId": "constructor_id",
            "name": "constructor_name"})
)

In [0]:
display(constructors_renamed_df)

####Step 5: Remove duplicate records

In [0]:
constructors_distinct_df = constructors_renamed_df.dropDuplicates(["constructor_id"])

In [0]:
display(constructors_distinct_df)

####Step 6: Transform the values of nationality to Title Case

In [0]:
constructors_final_df = (
    constructors_distinct_df
        .withColumn("nationality", F.initcap(F.col("nationality")))
)

display(constructors_final_df)

constructor_id,constructor_name,nationality,ingestion_timestamp,source_file,batch_id
adams,Adams,American,2026-08-02T16:46:32.036Z,dbfs:/Volumes/formula1_incr/landing/files/2025-01/constructors.json,2025-01
afm,AFM,German,2026-08-02T16:46:32.036Z,dbfs:/Volumes/formula1_incr/landing/files/2025-01/constructors.json,2025-01
ags,AGS,French,2026-08-02T16:46:32.036Z,dbfs:/Volumes/formula1_incr/landing/files/2025-01/constructors.json,2025-01
alfa,Alfa Romeo,Swiss,2026-08-02T16:46:32.036Z,dbfs:/Volumes/formula1_incr/landing/files/2025-01/constructors.json,2025-01
alphatauri,AlphaTauri,Italian,2026-08-02T16:46:32.036Z,dbfs:/Volumes/formula1_incr/landing/files/2025-01/constructors.json,2025-01
alpine,Alpine F1 Team,French,2026-08-02T16:46:32.036Z,dbfs:/Volumes/formula1_incr/landing/files/2025-01/constructors.json,2025-01
alta,Alta,British,2026-08-02T16:46:32.036Z,dbfs:/Volumes/formula1_incr/landing/files/2025-01/constructors.json,2025-01
amon,Amon,New Zealander,2026-08-02T16:46:32.036Z,dbfs:/Volumes/formula1_incr/landing/files/2025-01/constructors.json,2025-01
apollon,Apollon,Swiss,2026-08-02T16:46:32.036Z,dbfs:/Volumes/formula1_incr/landing/files/2025-01/constructors.json,2025-01
arrows,Arrows,British,2026-08-02T16:46:32.036Z,dbfs:/Volumes/formula1_incr/landing/files/2025-01/constructors.json,2025-01


####Step 7: Write the transformed data to the silver 'constructors' table

In [0]:
write_to_silver(
    input_df = constructors_final_df,
    target_table = silver_table,
    merge_condition = "t.constructor_id = s.constructor_id",
    columns_to_update = [
        "constructor_id",
        "constructor_name",
        "nationality",
        "ingestion_timestamp",
        "source_file",
        "batch_id"
    ]
)

In [0]:
display(spark.table(silver_table))

constructor_id,constructor_name,nationality,ingestion_timestamp,source_file,batch_id,created_timestamp,updated_timestamp
ats,ATS,Italian,2026-08-02T16:46:32.036Z,dbfs:/Volumes/formula1_incr/landing/files/2025-01/constructors.json,2025-01,2026-08-05T15:04:31.086Z,2026-08-05T15:04:54.464Z
benetton,Benetton,Italian,2026-08-02T16:46:32.036Z,dbfs:/Volumes/formula1_incr/landing/files/2025-01/constructors.json,2025-01,2026-08-05T15:04:31.086Z,2026-08-05T15:04:54.464Z
bmw,BMW,German,2026-08-02T16:46:32.036Z,dbfs:/Volumes/formula1_incr/landing/files/2025-01/constructors.json,2025-01,2026-08-05T15:04:31.086Z,2026-08-05T15:04:54.464Z
brabham-repco,Brabham-Repco,British,2026-08-02T16:46:32.036Z,dbfs:/Volumes/formula1_incr/landing/files/2025-01/constructors.json,2025-01,2026-08-05T15:04:31.086Z,2026-08-05T15:04:54.464Z
cadillac,Cadillac F1 Team,American,2026-08-02T16:46:32.036Z,dbfs:/Volumes/formula1_incr/landing/files/2025-01/constructors.json,2025-01,2026-08-05T15:04:31.086Z,2026-08-05T15:04:54.464Z
force_india,Force India,Indian,2026-08-02T16:46:32.036Z,dbfs:/Volumes/formula1_incr/landing/files/2025-01/constructors.json,2025-01,2026-08-05T15:04:31.086Z,2026-08-05T15:04:54.464Z
lotus-pw,Lotus-Pratt & Whitney,British,2026-08-02T16:46:32.036Z,dbfs:/Volumes/formula1_incr/landing/files/2025-01/constructors.json,2025-01,2026-08-05T15:04:31.086Z,2026-08-05T15:04:54.464Z
osella,Osella,Italian,2026-08-02T16:46:32.036Z,dbfs:/Volumes/formula1_incr/landing/files/2025-01/constructors.json,2025-01,2026-08-05T15:04:31.086Z,2026-08-05T15:04:54.464Z
token,Token,British,2026-08-02T16:46:32.036Z,dbfs:/Volumes/formula1_incr/landing/files/2025-01/constructors.json,2025-01,2026-08-05T15:04:31.086Z,2026-08-05T15:04:54.464Z
amon,Amon,New Zealander,2026-08-02T16:46:32.036Z,dbfs:/Volumes/formula1_incr/landing/files/2025-01/constructors.json,2025-01,2026-08-05T15:04:31.086Z,2026-08-05T15:04:54.464Z
